# 01 - Generate Synthetic Agreement Data

This notebook generates synthetic contract PDFs **with amendments** and simulates scanned documents.

## Pipeline
1. Generate clean PDFs (5 base contracts + 4 amendments, EN and PL)
2. Apply scan simulation (rotation, noise, contrast reduction)
3. Verify outputs

## Amendment-Aware Design
Amendments override specific clauses from the base contract. The filename convention
encodes metadata for ingestion: `{nn}_{contract_id}_{source_type}_{lang}_v{version}_{effective_date}.pdf`

In [1]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path

## Step 1: Generate Agreement PDFs with Amendments

We use `fpdf2` with DejaVu Sans font (Unicode support for Polish characters).

**Base contracts (5):**
| Contract ID | Type | Language | Effective Date |
|---|---|---|---|
| ITSVC-001 | IT Service Agreement | EN | 2025-01-15 |
| NDA-001 | Mutual NDA | EN | 2025-03-01 |
| LEASE-001 | Office Lease | PL | 2025-02-10 |
| SLA-001 | Cloud SLA | EN | 2025-04-01 |
| EMP-001 | Employment Contract | PL | 2025-05-01 |

**Amendments (4):**
| Contract ID | Amendment | Effective Date | Key Changes |
|---|---|---|---|
| ITSVC-001 | v2 | 2025-07-01 | Rate increase (+15%), team 3→5 |
| ITSVC-001 | v3 | 2026-01-01 | Rate increase (+10%), new AI/ML tier, bi-weekly invoicing |
| LEASE-001 | v2 | 2025-09-01 | Rent 25k→28.5k PLN, 5 parking spaces |
| SLA-001 | v2 | 2025-10-01 | P1 response 15→5 min, credit cap 30→50% |

In [2]:
%run ../scripts/generate_agreements.py

  Generated: 01_ITSVC-001_base_en_v1_2025-01-15.pdf (IT Service Agreement - base (EN))
  Generated: 02_ITSVC-001_amendment_en_v2_2025-07-01.pdf (IT Service Agreement - amendment 1 (EN))


  Generated: 03_ITSVC-001_amendment_en_v3_2026-01-01.pdf (IT Service Agreement - amendment 2 (EN))
  Generated: 04_NDA-001_base_en_v1_2025-03-01.pdf (Mutual NDA (EN))


  Generated: 05_LEASE-001_base_pl_v1_2025-02-10.pdf (Office Lease - base (PL))
  Generated: 06_LEASE-001_amendment_pl_v2_2025-09-01.pdf (Office Lease - amendment 1 (PL))
  Generated: 07_SLA-001_base_en_v1_2025-04-01.pdf (Cloud SLA - base (EN))


  Generated: 08_SLA-001_amendment_en_v2_2025-10-01.pdf (Cloud SLA - amendment 1 (EN))
  Generated: 09_EMP-001_base_pl_v1_2025-05-01.pdf (Employment Contract (PL))

All 9 documents generated in /home/kuba/repos/contract-lens-with-llamaindex/data/agreements
  Base contracts: 5
  Amendments:     4


In [3]:
# Verify generated files
agreements = sorted(Path('../data/agreements').glob('*.pdf'))
for p in agreements:
    print(f'{p.name:45s} {p.stat().st_size / 1024:.1f} KB')

01_ITSVC-001_base_en_v1_2025-01-15.pdf        34.9 KB
02_ITSVC-001_amendment_en_v2_2025-07-01.pdf   31.6 KB
03_ITSVC-001_amendment_en_v3_2026-01-01.pdf   31.9 KB
04_NDA-001_base_en_v1_2025-03-01.pdf          31.2 KB
05_LEASE-001_base_pl_v1_2025-02-10.pdf        36.9 KB
06_LEASE-001_amendment_pl_v2_2025-09-01.pdf   33.8 KB
07_SLA-001_base_en_v1_2025-04-01.pdf          34.9 KB
08_SLA-001_amendment_en_v2_2025-10-01.pdf     32.5 KB
09_EMP-001_base_pl_v1_2025-05-01.pdf          36.0 KB


## Step 2: Simulate Scanned Documents

Each PDF is converted to images and degraded to look like a scan:
- Slight rotation (±1.5°)
- Gaussian noise
- Reduced contrast and brightness
- Subtle blur

In [4]:
%run ../scripts/simulate_scans.py

  Scanned: 01_ITSVC-001_base_en_v1_2025-01-15.pdf


  Scanned: 02_ITSVC-001_amendment_en_v2_2025-07-01.pdf


  Scanned: 03_ITSVC-001_amendment_en_v3_2026-01-01.pdf


  Scanned: 04_NDA-001_base_en_v1_2025-03-01.pdf


  Scanned: 05_LEASE-001_base_pl_v1_2025-02-10.pdf


  Scanned: 06_LEASE-001_amendment_pl_v2_2025-09-01.pdf


  Scanned: 07_SLA-001_base_en_v1_2025-04-01.pdf


  Scanned: 08_SLA-001_amendment_en_v2_2025-10-01.pdf


  Scanned: 09_EMP-001_base_pl_v1_2025-05-01.pdf

All 9 scans generated in /home/kuba/repos/contract-lens-with-llamaindex/data/scans


In [5]:
# Verify scanned files (should be much larger - image-based PDFs)
scans = sorted(Path('../data/scans').glob('*.pdf'))
for p in scans:
    print(f'{p.name:45s} {p.stat().st_size / 1024:.1f} KB')

01_ITSVC-001_base_en_v1_2025-01-15.pdf        1650.6 KB
02_ITSVC-001_amendment_en_v2_2025-07-01.pdf   626.2 KB
03_ITSVC-001_amendment_en_v3_2026-01-01.pdf   660.3 KB
04_NDA-001_base_en_v1_2025-03-01.pdf          1145.4 KB
05_LEASE-001_base_pl_v1_2025-02-10.pdf        1729.2 KB
06_LEASE-001_amendment_pl_v2_2025-09-01.pdf   630.4 KB
07_SLA-001_base_en_v1_2025-04-01.pdf          1230.1 KB
08_SLA-001_amendment_en_v2_2025-10-01.pdf     1147.5 KB
09_EMP-001_base_pl_v1_2025-05-01.pdf          1696.0 KB


## Summary

We now have:
- `data/agreements/` — 9 clean PDFs (5 bases + 4 amendments)
- `data/scans/` — 9 scan-simulated PDFs ready for OCR/ingestion

Each filename encodes: `contract_id`, `source_type`, `language`, `version`, `effective_date`.

**Next:** Run `02_ingestion.ipynb` to embed and index these documents in Pinecone.